In [1]:
from coffea.nanoevents import NanoAODSchema
from coffea.dataset_tools import apply_to_fileset, max_chunks, max_files, preprocess

import dask
from dask.distributed import Client

from analysis_tools.processors.processor_17_2_25 import SignalProcessor
#The old skim_signal_processor is probably renamed and in the old_nonworking folder. 

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector


In [2]:
import gzip
import json
import os

# Define the base directory where the preprocessed files are stored
base_dir = "/home/cms-jovyan/dwg_analysis/tools/preprocessing/preprocessed"
signal_sample = "2023_SlepSnu_MN1_270_100000_preprocessed_available.json.gz"
#signal_sample = "2023_ttbar_100000_preprocessed_available.json.gz"
background_sample = "2023_ttbar_100000_preprocessed_available.json.gz"
signal_file_path = os.path.join(base_dir, signal_sample)
background_file_path = os.path.join(base_dir, background_sample)
#print(preprocessed)

with gzip.open(signal_file_path, "rt") as file:
    signal_preprocessed_available = json.load(file)
with gzip.open(background_file_path, "rt") as file:
    background_preprocessed_available = json.load(file)

In [3]:
client = Client("tls://localhost:8786")
client

/usr/local/lib/python3.12/site-packages/distributed/client.py:1613: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 1.26.4 | 1.26.4    | 2.1.3   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


<Client: 'tls://192.168.235.66:8786' processes=1 threads=1, memory=2.89 GiB>

In [4]:
signal_test_preprocessed_files = max_files(signal_preprocessed_available, 5)
signal_test_preprocessed = max_chunks(signal_test_preprocessed_files, 1)

### SWITCH HERE ###

signal_reduced_computation = False

###################

# signal

if signal_reduced_computation:
    small_tg, small_rep = apply_to_fileset(
        data_manipulation=SignalProcessor(),
        fileset=signal_test_preprocessed,
        schemaclass=NanoAODSchema,
        uproot_options={"allow_read_errors_with_report": (OSError, KeyError)},
    )
    signal_computed, rep = dask.compute(small_tg, small_rep)
    
else:
    full_tg, full_rep = apply_to_fileset(
        data_manipulation=SignalProcessor(),
        fileset=signal_preprocessed_available,
        schemaclass=NanoAODSchema,
        uproot_options={"allow_read_errors_with_report": (OSError, KeyError)},
    )
    signal_computed, rep = dask.compute(full_tg, full_rep)


'total_ele': 90844,
   'events_filtered': 16498,
   'count_filtered': 16765,
   'events_baseline': 12606,
   'count_baseline': 12816,
   'events_blp': 7803,
   'count_blp': 7912,
   'events_bronze': 1397,
   'count_bronze': 1403,
   'events_silver': 856,
   'count_silver': 857,
   'events_gold': 5595,
   'count_gold': 5652},
  'pt_binned': {},
  'calculations': {},

In [5]:
signal_computed

{'/SlepSnuCascade_MN1-270_MN2-280_MC1-275_TuneCP5_13p6TeV_madgraphMLM-pythia8/Run3Summer23BPixNanoAODv12-130X_mcRun3_2023_realistic_postBPix_v6-v3/NANOAODSIM': {'counts': {'total_entries': 459524,
   'total_ele': 90844,
   'events_filtered': 16498,
   'count_filtered': 16765,
   'events_baseline': 12606,
   'count_baseline': 12816,
   'events_blp': 7803,
   'count_blp': 7912,
   'events_bronze': 1397,
   'count_bronze': 1403,
   'events_silver': 856,
   'count_silver': 857,
   'events_gold': 5595,
   'count_gold': 5652},
  'pt_binned': {},
  'calculations': {},
  'plots': {'filtered_plots': {'filtered_pt': Hist(Regular(100, 0, 50, underflow=False, overflow=False, name='pT'), storage=Double()) # Sum: 16765.0,
    'filtered_eta': Hist(Regular(100, -3, 3, name='eta'), storage=Double()) # Sum: 16765.0,
    'filtered_pt_vs_mini_iso': Hist(
      Regular(100, 5, 30, name='pT'),
      Regular(100, 0, 10, name='mini_iso_pt'),
      storage=Double()) # Sum: 16398.0 (16765.0 with flow),
    'fil

In [6]:
#sig_results = signal_computed['/SlepSnuCascade_MN1-270_MN2-280_MC1-275_TuneCP5_13p6TeV_madgraphMLM-pythia8/Run3Summer23BPixNanoAODv12-130X_mcRun3_2023_realistic_postBPix_v6-v3/NANOAODSIM']
sig_results = signal_computed['/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/Run3Summer23NanoAODv12-130X_mcRun3_2023_realistic_v14-v2/NANOAODSIM']

KeyError: '/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/Run3Summer23NanoAODv12-130X_mcRun3_2023_realistic_v14-v2/NANOAODSIM'

Shooting for ~ 459524 events for ttbar

In [ ]:
sig_results

In [ ]:
sig_counts = sig_results["counts"]
sig_counts

In [ ]:
sig_test = sig_results["tests"]
sig_test

In [ ]:
sig_plots = sig_results["plots"]


{'counts': {'count_ele': 228295,
  'count_filtered_ele': 48104,
  'count_baseline_ele': 143846,
  'count_baseline_plus_ele': 103464,
  'count_bronze_ele': 21497},
 'calculations': {},
 'plots': {},
 'plots_2d': {},
 'tests': {}}

before changing bronze mask to be more inclusive, should now see more bronze electrons than whats below:
{'counts': {'total_entries': 459524,
  'total_ele': 345260,
  'events_filtered': 69914,
  'count_filtered': 72844,
  'events_baseline': 57408,
  'count_baseline': 59593,
  'events_baseline_p': 45216,
  'count_baseline_p': 46634,
  'events_bronze': 9786,
  'count_bronze': 9941,
  'events_silver': 4721,
  'count_silver': 4726,
  'events_gold': 29588,
  'count_gold': 30054},
 'pt_binned': {},
 'calculations': {},
 'plots': {},
 'plots_2d': {},
 'tests': {'test_dr': <Array [[0], [], [0, 0], ..., [0], [0, 0]] type='459524 * var * ?float32'>}}

In [ ]:
#have to call this a second time to get proper scaling, a known bug
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import mplhep
mplhep.style.use(mplhep.style.CMS)
plt.figure()
mplhep.style.use(mplhep.style.CMS)

In [ ]:

def plot_electron_histograms(hist_dict, label="default"):
    """
    Plots histograms from a dictionary of histograms with fixed log-scale y-axes.
    Displays total counts in each plot.

    Parameters:
    - hist_dict: Dictionary containing histograms (from make_electron_histograms)
    - label: String label for plot titles (default="filtered")

    Returns:
    - Displays histograms
    """

    # Define a figure layout
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))  # 2 rows, 3 columns
    mplhep.cms.label(loc=0)

    # Define max values for y-axis log scales and color axis:
    y_min, y_max = 1, 1e4
    color_min, color_max = 1, 1e3

    # Define histogram keys and plot titles
    hist_keys = [
        f"{label}_pt",
        f"{label}_eta",
        f"{label}_pt_vs_mini_iso",
        f"{label}_pt_vs_pfrel_iso",
        f"{label}_pt_vs_sip3d",
    ]

    titles = [
        r"Electron $p_T$ Distribution",
        "Electron Eta Distribution",
        r"Electron $p_T$ vs MiniIso*$p_T$",
        r"Electron $p_T$ vs PFRelIso*$p_T$",
        r"Electron $p_T$ vs SIP3D",
    ]

    # Iterate over histogram keys and plot them
    for ax, hist_key, title in zip(axes.flat, hist_keys, titles):
        hist = hist_dict.get(hist_key)
        if hist is None:
            continue  # Skip if histogram is missing

        # Calculate total counts
        total_counts = hist.values().sum()

        # 1D histograms (pT and eta)
        if "vs" not in hist_key:
            ax.bar(hist.axes[0].centers, hist.values(), width=hist.axes[0].widths)
            ax.set_xlabel(hist_key.split("_")[-1])
            ax.set_ylabel("Counts")
            ax.set_yscale("log")  # ✅ Set log scale for y-axis
            ax.set_ylim(y_min, y_max)  # ✅ Fixed y-axis range
            
            # Add text for total counts
            ax.text(0.95, 0.95, f"Total: {int(total_counts):,}",
                    transform=ax.transAxes, fontsize=12, 
                    verticalalignment='top', horizontalalignment='right',
                    bbox=dict(facecolor='white', alpha=0.8))

        # 2D histograms (pT vs MiniIso, PFRelIso, SIP3D)
        else:
            pcm = ax.pcolormesh(
                hist.axes[0].edges,
                hist.axes[1].edges,
                hist.values().T,
                shading="auto",
                cmap="viridis",
                norm=colors.LogNorm(vmin=color_min, vmax=color_max)
            )
            fig.colorbar(pcm, ax=ax)
            ax.set_xlabel(f"{hist.axes[0].name} (GeV)")
            ax.set_ylabel(hist.axes[1].name)

            # Add text for total counts
            ax.text(0.95, 0.95, f"Total: {int(total_counts):,}",
                    transform=ax.transAxes, fontsize=14, 
                    verticalalignment='top', horizontalalignment='right',
                    bbox=dict(facecolor='white', alpha=0.8))

        ax.set_title(title)
        ax.grid(True)

    plt.tight_layout()
    plt.savefig(f"{label}_plots")
    plt.show()

In [ ]:
sig_plots.keys()

In [ ]:

plot_electron_histograms(sig_plots['filtered_plots'], "filtered")
plot_electron_histograms(sig_plots['baseline_plots'], "baseline")
plot_electron_histograms(sig_plots['blp_plots'], "blp")
plot_electron_histograms(sig_plots['bronze_plots'], "bronze")
plot_electron_histograms(sig_plots['silver_plots'], "silver")
plot_electron_histograms(sig_plots['gold_plots'], "gold")
